In [ ]:
CONFIG = {
    # ---- cities -----------------------------------------------------------
    # Training cities and hold-out cities MUST be different — that is exactly
    # what proves generalization to unseen cities. Add any city on Earth.
    "TRAIN_CITIES": ["Kazan, Russia", "Yekaterinburg, Russia"],
    "UNSEEN_CITIES": ["Perm, Russia"],          # never used for training

    # ---- imagery / fixed scale --------------------------------------------
    "IMAGERY_SOURCE": "esri",       # "esri" (satellite) | "osm" (drawn map, see warning above)
    "TARGET_M_PER_PIXEL": 0.7,      # FIXED ground scale; zoom chosen per-city latitude
                                    # so that all cities are at the same scale.
                                    # Set to None to use fixed FALLBACK_ZOOM instead.
    "FALLBACK_ZOOM": 17,            # used if TARGET_M_PER_PIXEL is None
    "TILE_SIZE": 512,               # must be a multiple of 256

    # ---- road labels --------------------------------------------------------
    # "geofabrik_pbf" (recommended): pre-packaged OSM region extract from
    #   Geofabrik's CDN — static files, no rate limits, no query timeouts.
    #   The region containing the city is resolved automatically from the
    #   official index, so ANY city on Earth works.
    # "overpass": legacy live-API source (kept as fallback; big-city queries
    #   often time out on every public mirror).
    "ROADS_SOURCE": "geofabrik_pbf",
    "GEOFABRIK_INDEX_URL": "https://download.geofabrik.de/index-v1.json",
    "GEOFABRIK_BBOX_MARGIN_DEG": 0.02,  # ~2 km margin: keep roads that enter the city
    "PBF_MAX_DOWNLOAD_MB": 2500,        # refuse absurdly large region files
    "CACHE_DIR": None,                  # None => <system tmp>/roadseg_cache
                                        # (pbf downloads reused between runs)

    # ---- Overpass fallback endpoints (tried in order) -----------------------
    "OVERPASS_ENDPOINTS": [
        "https://overpass-api.de/api/interpreter",
        "https://overpass.kumi.systems/api/interpreter",
        "https://overpass.private.coffee/api/interpreter",
        "https://overpass.osm.ch/api/interpreter",
    ],
    "OVERPASS_TIMEOUT_S": 90,       # per-endpoint client timeout

    # ---- data volume ------------------------------------------------------
    "MAX_TILES_PER_CITY": 450,      # even grid coverage across the city
    "MIN_ROAD_FRACTION": 0.01,      # drop tiles with less road than this
    "MAX_EMPTY_TILES": 40,          # ...but keep some road-free hard negatives

    # ---- road label geometry (OSM highway classes) -------------------------
    # NOTE: the default geofabrik_pbf source takes its class list from the
    # ROAD_WIDTH_M keys below; ROAD_HIGHWAY_TYPES is only used by the
    # legacy Overpass fallback. Add new classes to BOTH to be safe.
    "ROAD_HIGHWAY_TYPES": [
        "motorway", "trunk", "primary", "secondary", "tertiary",
        "unclassified", "residential", "living_street", "service",
        "motorway_link", "trunk_link", "primary_link", "secondary_link",
        "tertiary_link",
    ],
    "ROAD_WIDTH_M": {               # approximate real-world painted width in meters
        "motorway": 18.0, "trunk": 14.0, "primary": 11.0, "secondary": 9.0,
        "tertiary": 7.5, "unclassified": 6.5, "residential": 6.0,
        "living_street": 5.0, "service": 4.0,
        "motorway_link": 7.0, "trunk_link": 6.5, "primary_link": 6.0,
        "secondary_link": 5.5, "tertiary_link": 5.0,
    },
    "DEFAULT_ROAD_WIDTH_M": 4.0,

    # ---- training ----------------------------------------------------------
    "VAL_SPLIT": 0.10,
    "BATCH_SIZE": 8,
    "MAX_EPOCHS": 80,               # ~20 min on T4 for 2 cities; TRAIN_TIME_BUDGET_MIN
                                    # still stops earlier if needed (30 epochs = undertrained)
    "TRAIN_TIME_BUDGET_MIN": 40,    # ~1h total on a free T4 incl. data download
    "LR": 1e-3,
    "PRETRAINED_ENCODER": True,     # ResNet-18 encoder with ImageNet weights: converges in a
                                    # few epochs instead of dozens — the single biggest speed-up
                                    # at ~1k tiles. Falls back to the compact from-scratch U-Net
                                    # (UNET_BASE) if torchvision or the weight download fails.
    "ENCODER_LR": 2.5e-4,           # LR for pretrained encoder weights (fresh decoder uses LR)
    "WEIGHT_DECAY": 1e-4,
    "BCE_POS_WEIGHT": "auto",       # "auto" = balance road/background per batch (up to 20);
                                    # or a float, e.g. 8.0 — LOWER it if roads are over-marked
    "LOSS_MODE": "tversky_bce",     # "tversky_bce": the overlap term punishes FALSE POSITIVES
                                    #   (forests/houses painted as roads) harder than missed
                                    #   roads — recommended against over-marking.
                                    # "dice_bce": previous behaviour (balanced Dice)
    "TVERSKY_ALPHA": 0.7,           # false-positive weight (raise to 0.8 for harsher FP cost)
    "TVERSKY_BETA": 0.3,            # false-negative weight (alpha=beta=0.5 reproduces Dice)
    "OVERLAP_WEIGHT": 1.0,          # weight of the Dice/Tversky term in the total loss
    "AMP": True,                    # mixed precision on GPU
    "NUM_WORKERS": 2,
    "SEED": 42,
    "UNET_BASE": 16,                # first-layer channels (32 => ~4x params)

    # ---- inference on unseen cities ---------------------------------------
    "STITCH_RADIUS_TILES": 3,       # demo image = (2r+2) x (2r+2) 256-tiles => 2048 px wide at r=3
    "OVERLAY_ALPHA": 0.45,          # red road paint opacity on the photo
    "PRED_THRESHOLD": 0.40,         # paint threshold on the demo overlay (IoU eval stays 0.5);
                                    # lower it to 0.30 to mark more (thinner) roads, raise to
                                    # 0.5+ if the demo looks over-painted / noisy
    "PRED_TTA": True,               # average predictions over 4 rotations of every window:
                                    # recovers thin roads one orientation misses, ~4x slower
    "PRED_BATCH": 8,

    # ---- outputs -----------------------------------------------------------
    "SAVE_DIR": "/content/road_seg_outputs",   # on Colab; local run saves here too
}

# =============================================================================
# 1. Environment: auto-install, imports, seeds, device
# =============================================================================
import importlib.util
import json
import math
import os
import random
import shutil
import subprocess
import sys
import tempfile
import time
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import requests

requests.packages.urllib3.disable_warnings()


def _ensure_package(import_name: str, pip_name: str | None = None) -> None:
    """Import if possible, otherwise pip-install it (Colab-friendly)."""
    if importlib.util.find_spec(import_name) is None:
        pkg = pip_name or import_name
        print(f"[setup] installing missing package: {pkg} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", pkg])
    # import check
    mod = importlib.import_module(import_name)
    assert mod is not None


_ensure_package("PIL", "pillow")
_ensure_package("matplotlib")
_ensure_package("shapely")
_ensure_package("osmium")        # pyosmium: C++ reader for .osm.pbf extracts
_ensure_package("tqdm")

try:
    import torch
except ImportError:  # torch is preinstalled on Colab; helpful message elsewhere
    raise ImportError(
        "PyTorch not found. On Colab it is preinstalled. "
        "Locally: pip install torch"
    )

import matplotlib
if "ipykernel" not in sys.modules:      # plain script run -> non-interactive backend
    matplotlib.use("Agg")
import matplotlib.pyplot as plt

import shapely
from shapely.geometry import LineString, Point, box, shape
from shapely.prepared import prep
from PIL import Image, ImageDraw
from tqdm.auto import tqdm

import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True

# =============================================================================
# 2. Geo / Web-Mercator tile math  (slippy-map standard, EPSG:3857)
# =============================================================================
EARTH_R = 6378137.0
ORIGIN = math.pi * EARTH_R          # half world width in Mercator meters
TILE = 256                          # standard tile size in px


def deg2num(lat_deg: float, lon_deg: float, zoom: int) -> tuple[float, float]:
    """lat/lon -> fractional (xtile, ytile) on the 256-px slippy grid."""
    n = 2 ** zoom
    lat = math.radians(max(min(lat_deg, 85.05112878), -85.05112878))
    x = (lon_deg + 180.0) / 360.0 * n
    y = (1.0 - math.log(math.tan(lat) + 1.0 / math.cos(lat)) / math.pi) / 2.0 * n
    return x, y


def num2deg(x: float, y: float, zoom: int) -> tuple[float, float]:
    """fractional (xtile, ytile) -> lat/lon of the tile's NW corner."""
    n = 2 ** zoom
    lon = x / n * 360.0 - 180.0
    lat = math.degrees(math.atan(math.sinh(math.pi * (1.0 - 2.0 * y / n))))
    return lat, lon


def merc_x(lon_deg: float) -> float:
    return EARTH_R * math.radians(lon_deg)


def merc_y(lat_deg: float) -> float:
    return EARTH_R * math.log(math.tan(math.pi / 4.0 + math.radians(lat_deg) / 2.0))


def lonlat_to_global_px(lat: float, lon: float, zoom: int) -> tuple[float, float]:
    """lat/lon -> global pixel coordinates (top-left origin) on the 256-px grid."""
    mx, my = merc_x(lon), merc_y(lat)
    px = (mx + ORIGIN) / (2.0 * ORIGIN) * (2 ** zoom) * TILE
    py = (ORIGIN - my) / (2.0 * ORIGIN) * (2 ** zoom) * TILE
    return px, py


def m_per_pixel(lat_deg: float, zoom: int) -> float:
    """Ground resolution (m/px) of a 256-px tile at this latitude and zoom."""
    return 156543.03392 * math.cos(math.radians(lat_deg)) / (2 ** zoom)


def pick_zoom(lat_deg: float, target_mpp: float | None, fallback: int = 17,
              zmin: int = 16, zmax: int = 19) -> int:
    """Choose the integer zoom whose ground resolution is closest to target.

    This is what keeps the FIXED IMAGE SCALE across cities at different
    latitudes (m/px depends on latitude for a constant zoom).
    """
    if target_mpp is None:
        return fallback
    return min(range(zmin, zmax + 1),
               key=lambda z: abs(m_per_pixel(lat_deg, z) - target_mpp))


def self_test_geo() -> None:
    """Fast consistency checks of the geo math (no network)."""
    rng = random.Random(0)
    for _ in range(200):
        lat = rng.uniform(-60, 70)
        lon = rng.uniform(-170, 170)
        z = rng.randint(3, 18)
        x, y = deg2num(lat, lon, z)
        lat2, lon2 = num2deg(x, y, z)                  # exact roundtrip
        assert abs(lat2 - lat) < 1e-6 and abs(lon2 - lon) < 1e-6, (lat, lon, z)
        # lon/lon -> mercator -> back
        mx, my = merc_x(lon), merc_y(lat)
        lon3 = math.degrees(mx / EARTH_R)
        lat3 = math.degrees(2.0 * math.atan(math.exp(my / EARTH_R)) - math.pi / 2.0)
        assert abs(lon3 - lon) < 1e-9 and abs(lat3 - lat) < 1e-9
        # global pixel == fractional tile * 256
        px, py = lonlat_to_global_px(lat, lon, z)
        assert abs(px / TILE - x) < 1e-6 and abs(py / TILE - y) < 1e-6
    # plausible resolutions
    assert 0.6 < m_per_pixel(55.8, 17) < 0.75
    assert pick_zoom(55.8, 0.7) == 17   # northern cities
    assert pick_zoom(0.0, 0.7) == 18    # equator needs one step up for same scale
    print("[self-test] geo math OK")


# =============================================================================
# 3. Data acquisition: city polygon, OSM roads, imagery tiles, mask raster
# =============================================================================
_UA = {"User-Agent": "road-segmentation-colab/1.0 (student project)"}
_session = requests.Session()
_session.headers.update(_UA)
_session.mount("https://", requests.adapters.HTTPAdapter(max_retries=2))

IMAGERY_URLS = {
    "esri": "https://server.arcgisonline.com/ArcGIS/rest/services/"
            "World_Imagery/MapServer/tile/{z}/{y}/{x}",
    "osm":  "https://tile.openstreetmap.org/{z}/{x}/{y}.png",
}


def get_city_polygon(city_name: str) -> shapely.geometry.base.BaseGeometry:
    """Geocode a city via Nominatim -> shapely Polygon/MultiPolygon.

    Falls back to a square around the bbox if the result is a point.
    """
    r = _session.get(
        "https://nominatim.openstreetmap.org/search",
        params={"q": city_name, "format": "jsonv2", "polygon_geojson": 1, "limit": 1},
        timeout=60,
    )
    r.raise_for_status()
    data = r.json()
    if not data:
        raise RuntimeError(f"Nominatim found nothing for: {city_name!r}")
    rec = data[0]
    geom = shape(rec["geojson"])
    if geom.geom_type == "Point" or not geom.geom_type.startswith(("Polygon", "Multi")):
        bb = rec["boundingbox"]  # [south, north, west, east]
        south, north, west, east = map(float, bb)
        geom = shapely.geometry.box(west, south, east, north)
    print(f"[geocode] {city_name!r} -> {geom.geom_type}, "
          f"bbox {tuple(round(b, 3) for b in geom.bounds)}")
    return geom


def download_tile(z: int, x: int, y: int, source: str = "esri") -> np.ndarray | None:
    """Download one 256x256 imagery tile -> HxWx3 uint8 (or None on failure)."""
    url = IMAGERY_URLS[source].format(z=z, x=x, y=y)
    try:
        r = _session.get(url, timeout=40)
        if r.status_code != 200 or len(r.content) < 500:
            return None
        from io import BytesIO
        img = Image.open(BytesIO(r.content)).convert("RGB")
        if img.size != (TILE, TILE):
            img = img.resize((TILE, TILE))
        return np.asarray(img, dtype=np.uint8)
    except Exception:
        return None


# ---- primary source: Geofabrik pre-packaged OSM extracts (.osm.pbf) -------

_geofabrik_index_cache: list[dict] | None = None


def _cache_dir() -> str:
    """Directory for cached index/pbf files (reused between runs)."""
    d = CONFIG.get("CACHE_DIR") or os.path.join(tempfile.gettempdir(), "roadseg_cache")
    os.makedirs(d, exist_ok=True)
    return d


def _load_geofabrik_index() -> list[dict]:
    """Download & cache Geofabrik's region index (~3 MB, 554 regions).

    Each feature carries the region polygon and the URL of its pre-packaged
    .osm.pbf extract (refreshed daily by Geofabrik on a static CDN)."""
    global _geofabrik_index_cache
    if _geofabrik_index_cache is not None:
        return _geofabrik_index_cache
    cache_file = os.path.join(_cache_dir(), "geofabrik_index.json")
    if not os.path.exists(cache_file):
        print("[geofabrik] downloading region index (~3 MB) ...")
        r = _session.get(CONFIG["GEOFABRIK_INDEX_URL"], timeout=120)
        r.raise_for_status()
        with open(cache_file, "wb") as f:
            f.write(r.content)
    with open(cache_file) as f:
        data = json.load(f)
    feats: list[dict] = []
    for feat in data["features"]:
        try:
            feats.append({
                "id": feat["properties"]["id"],
                "name": feat["properties"]["name"],
                "geom": shape(feat["geometry"]),
                "pbf": feat["properties"]["urls"]["pbf"],
            })
        except (KeyError, TypeError):
            continue
    _geofabrik_index_cache = feats
    return feats


def _resolve_geofabrik_region(polygon) -> tuple[str, str, str]:
    """Smallest Geofabrik region containing the city -> (id, name, pbf_url)."""
    feats = _load_geofabrik_index()
    minx, miny, maxx, maxy = polygon.bounds
    pt = polygon.representative_point()
    candidates = []
    for f in feats:
        gx0, gy0, gx1, gy1 = f["geom"].bounds
        if gx1 < minx or gx0 > maxx or gy1 < miny or gy0 > maxy:
            continue
        if f["geom"].intersects(polygon):
            candidates.append(f)
    if not candidates:
        raise RuntimeError(
            "No Geofabrik region extract covers this city — very remote place?")
    containing = [f for f in candidates if f["geom"].contains(pt)]
    if containing:
        pool = containing
    else:
        pool = candidates
        print("[geofabrik] warning: city straddles region borders; using the "
              "smallest intersecting region")
    best = min(pool, key=lambda f: f["geom"].area)
    return best["id"], best["name"], best["pbf"]


def _download_big_file(url: str, dest: str, max_mb: float) -> None:
    """Streamed download with progress prints; skips if already cached."""
    if os.path.exists(dest) and os.path.getsize(dest) > 1_000_000:
        print(f"[geofabrik] reusing cached {os.path.basename(dest)} "
              f"({os.path.getsize(dest) / 1e6:.0f} MB)")
        return
    head = _session.head(url, timeout=60, allow_redirects=True)
    total = int(head.headers.get("Content-Length", 0))
    if max_mb and total / 1e6 > max_mb:
        raise RuntimeError(
            f"Region extract is {total / 1e6:.0f} MB (> PBF_MAX_DOWNLOAD_MB="
            f"{max_mb}). Raise that CONFIG value or pick a city covered by "
            f"a smaller region.")
    print(f"[geofabrik] downloading {os.path.basename(dest)} "
          f"({total / 1e6:.0f} MB) ...")
    t0 = time.time()
    with _session.get(url, stream=True, timeout=900, allow_redirects=True) as r:
        r.raise_for_status()
        done = 0
        next_mark = 0.25
        with open(dest + ".part", "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)
                done += len(chunk)
                if total and done / total >= next_mark:
                    speed = done / 1e6 / max(1e-9, time.time() - t0)
                    print(f"           {done / total * 100:.0f}%  "
                          f"({done / 1e6:.0f} MB, {speed:.1f} MB/s)")
                    next_mark += 0.25
    os.replace(dest + ".part", dest)
    print(f"[geofabrik] download finished in {time.time() - t0:.0f}s")


def _osmium_clip(region_pbf: str, bbox: tuple[float, float, float, float],
                 out_pbf: str) -> str | None:
    """Clip the region extract to the city bbox with the osmium C++ tool.

    Returns the clipped file path, or None if osmium is unavailable (the
    pure-Python reader then scans the whole region file instead — slower
    but memory-safe)."""
    if shutil.which("osmium") is None:
        # Colab is a root Ubuntu box: apt just works.
        try:
            subprocess.run("apt-get install -y -qq osmium-tool", shell=True,
                           check=True, capture_output=True, timeout=300)
        except Exception:
            pass
    if shutil.which("osmium") is None:
        # some Colab images need an apt list refresh first
        try:
            subprocess.run(
                "apt-get update -qq && apt-get install -y -qq osmium-tool",
                shell=True, check=True, capture_output=True, timeout=600)
        except Exception:
            pass
    if shutil.which("osmium") is None:
        return None
    b = ",".join(f"{v:.6f}" for v in bbox)
    t0 = time.time()
    try:
        subprocess.run(
            ["osmium", "extract", "-b", b, "--strategy", "complete_ways",
             region_pbf, "-o", out_pbf, "--overwrite"],
            check=True, capture_output=True, timeout=1800)
    except Exception as e:
        print(f"[geofabrik] osmium extract failed ({type(e).__name__}) — "
              f"using the Python scan")
        return None
    print(f"[geofabrik] osmium clip -> {os.path.basename(out_pbf)} "
          f"({os.path.getsize(out_pbf) / 1e6:.1f} MB, {time.time() - t0:.0f}s)")
    return out_pbf


def _read_roads_from_pbf(pbf_path: str, bbox: tuple[float, float, float, float],
                         city_geom, zoom: int, mpp: float) -> list[dict]:
    """Two-pass pyosmium scan of an .osm.pbf -> road list (pixel coords).

    Pass 1 caches only node positions inside the (expanded) bbox, so RAM
    stays small even for multi-GB region files. Pass 2 rebuilds highway-way
    geometries from the cached nodes and keeps only lines inside the city
    polygon. Output format is identical to the Overpass path:
    [{pts: (K,2) global 256-grid pixels, width_m, width_px}, ...]
    """
    import osmium
    minx, miny, maxx, maxy = bbox
    widths = CONFIG["ROAD_WIDTH_M"]
    default_w = CONFIG["DEFAULT_ROAD_WIDTH_M"]
    cityp = prep(city_geom)
    n = 2 ** zoom

    t0 = time.time()
    loc: dict[int, tuple[float, float]] = {}
    for obj in osmium.FileProcessor(pbf_path):
        if obj.is_node():
            lon, lat = obj.location.lon, obj.location.lat
            if minx <= lon <= maxx and miny <= lat <= maxy:
                loc[obj.id] = (lon, lat)
    print(f"[roads] pass 1/2: cached {len(loc)} node positions "
          f"({time.time() - t0:.0f}s)")

    t0 = time.time()
    roads: list[dict] = []
    for obj in osmium.FileProcessor(pbf_path):
        if not obj.is_way():
            continue
        hw = obj.tags.get("highway")
        if hw not in widths:
            continue
        segments: list[list[tuple[float, float]]] = []
        seg: list[tuple[float, float]] = []
        for nd in obj.nodes:
            p = loc.get(nd.ref)
            if p is None:                     # node outside bbox -> cut here
                if len(seg) > 1:
                    segments.append(seg)
                seg = []
            else:
                seg.append(p)
        if len(seg) > 1:
            segments.append(seg)
        w_m = widths.get(hw, default_w)
        w_px = max(2, round(w_m / mpp))
        for seg in segments:
            line = LineString(seg)
            if not cityp.intersects(line):    # keep city polygon only
                continue
            arr = np.asarray(seg, dtype=np.float64)
            mx = EARTH_R * np.radians(arr[:, 0])
            my = EARTH_R * np.log(np.tan(np.pi / 4.0 + np.radians(arr[:, 1]) / 2.0))
            px = (mx + ORIGIN) / (2.0 * ORIGIN) * n * TILE
            py = (ORIGIN - my) / (2.0 * ORIGIN) * n * TILE
            roads.append({"pts": np.stack([px, py], axis=1),
                          "width_m": w_m, "width_px": w_px})
    print(f"[roads] pass 2/2: {len(roads)} road segments inside the city "
          f"({time.time() - t0:.0f}s)")
    return roads


def fetch_roads_geofabrik(polygon, mpp: float, zoom: int) -> list[dict]:
    """Primary road-label source: Geofabrik .osm.pbf extract (see header)."""
    minx, miny, maxx, maxy = polygon.bounds
    m = CONFIG["GEOFABRIK_BBOX_MARGIN_DEG"]
    bbox = (minx - m, miny - m, maxx + m, maxy + m)

    region_id, region_name, pbf_url = _resolve_geofabrik_region(polygon)
    print(f"[roads] source: geofabrik_pbf | region {region_id!r} ({region_name})")
    region_pbf = os.path.join(_cache_dir(),
                              f"geofabrik-{region_id}-latest.osm.pbf")
    _download_big_file(pbf_url, region_pbf, CONFIG["PBF_MAX_DOWNLOAD_MB"])

    clip_name = (f"cityclip-{region_id}-{bbox[0]:.3f}_{bbox[1]:.3f}_"
                 f"{bbox[2]:.3f}_{bbox[3]:.3f}.osm.pbf")
    clipped = _osmium_clip(region_pbf, bbox, os.path.join(_cache_dir(), clip_name))
    if clipped is not None:
        roads = _read_roads_from_pbf(clipped, bbox, polygon, zoom, mpp)
    else:
        print("[roads] osmium not available — Python-scanning the whole "
              "region file (memory-safe but slower for huge regions)")
        roads = _read_roads_from_pbf(region_pbf, bbox, polygon, zoom, mpp)
    if not roads:
        raise RuntimeError("No OSM roads found in the extract — check the city name.")
    return roads


def fetch_osm_roads(polygon, mpp: float, zoom: int) -> list[dict]:
    """Road lines for the city polygon, source per CONFIG["ROADS_SOURCE"]."""
    if CONFIG.get("ROADS_SOURCE", "geofabrik_pbf") == "geofabrik_pbf":
        try:
            return fetch_roads_geofabrik(polygon, mpp, zoom)
        except Exception as e:
            print(f"[roads] geofabrik_pbf failed: {type(e).__name__}: {e}")
            print("[roads] falling back to the legacy Overpass API ...")
    return fetch_roads_overpass(polygon, mpp, zoom)


def fetch_roads_overpass(polygon, mpp: float, zoom: int) -> list[dict]:
    """Legacy source: live Overpass API via osmnx (see header for caveats).

    Tries every endpoint in CONFIG["OVERPASS_ENDPOINTS"] until one answers
    with actual data (some mirrors only cover their own region).
    Returns a list of {pts: (K,2) float array of GLOBAL 256-grid pixels,
    width_m: float, width_px: int}.
    """
    _ensure_package("osmnx")          # pulls geopandas/pyproj on Colab (~1 min)
    import osmnx as ox
    tags = {"highway": CONFIG["ROAD_HIGHWAY_TYPES"]}
    ox.settings.requests_timeout = CONFIG.get("OVERPASS_TIMEOUT_S", 90)
    gdf = None
    last_err = None
    for endpoint in CONFIG["OVERPASS_ENDPOINTS"]:
        try:
            print(f"[osm] trying Overpass endpoint: {endpoint}")
            ox.settings.overpass_url = endpoint
            gdf = ox.features_from_polygon(polygon, tags)
            if len(gdf) == 0:
                print("[osm] endpoint answered but returned no data (maybe "
                      "a regional mirror) — trying next one")
                gdf = None
                continue
            break
        except Exception as e:                      # network / rate-limit issues
            last_err = e
            print(f"[osm] endpoint failed: {type(e).__name__}")
    if gdf is None or len(gdf) == 0:
        raise RuntimeError(
            f"Could not fetch OSM roads: all Overpass endpoints failed. "
            f"Last error: {last_err}")
    gdf = gdf[gdf.geometry.notna() & (gdf.geometry.geom_type.isin(
        ["LineString", "MultiLineString"]))].copy()
    if gdf.empty:
        raise RuntimeError("Overpass returned no roads — check the city name / network.")

    widths = CONFIG["ROAD_WIDTH_M"]
    default_w = CONFIG["DEFAULT_ROAD_WIDTH_M"]
    roads: list[dict] = []
    for _, row in gdf.iterrows():
        hw = row.get("highway")
        if isinstance(hw, (list, tuple)):
            w_m = max((widths.get(h, default_w) for h in hw), default=default_w)
        else:
            w_m = widths.get(hw, default_w)
        geom = row.geometry
        if geom.geom_type == "MultiLineString":
            parts = list(geom.geoms)
        else:
            parts = [geom]
        for line in parts:
            coords = np.asarray(line.coords, dtype=np.float64)
            if len(coords) < 2:
                continue
            # lon/lat -> global pixel on the 256 grid
            lon = coords[:, 0]
            lat = coords[:, 1]
            n = 2 ** zoom
            mx = EARTH_R * np.radians(lon)
            my = EARTH_R * np.log(np.tan(np.pi / 4.0 + np.radians(lat) / 2.0))
            px = (mx + ORIGIN) / (2.0 * ORIGIN) * n * TILE
            py = (ORIGIN - my) / (2.0 * ORIGIN) * n * TILE
            pts = np.stack([px, py], axis=1)
            roads.append({"pts": pts, "width_m": w_m,
                          "width_px": max(2, round(w_m / mpp))})
    print(f"[osm] roads: {len(gdf)} ways -> {len(roads)} line parts")
    return roads


def enumerate_sample_anchors(polygon, zoom: int, max_tiles: int) -> list[tuple[int, int]]:
    """Even grid of (tx, ty) 256-tile anchors of TILE_SIZE blocks whose centre
    is inside the city polygon; stride-sampled to ~max_tiles, then shuffled."""
    minx, miny, maxx, maxy = polygon.bounds
    x0, y0 = deg2num(maxy, minx, zoom)          # NW corner
    x1, y1 = deg2num(miny, maxx, zoom)          # SE corner
    prep_poly = prep(polygon)
    candidates: list[tuple[int, int]] = []
    for ty in range(int(math.floor(y0)), int(math.floor(y1)) + 1, 2):
        for tx in range(int(math.floor(x0)), int(math.floor(x1)) + 1, 2):
            lat_c, lon_c = num2deg(tx + 1, ty + 1, zoom)  # centre of the 512 block
            if prep_poly.contains(Point(lon_c, lat_c)):
                candidates.append((tx, ty))
    if not candidates:
        raise RuntimeError("No city tiles inside polygon — polygon too small?")
    if len(candidates) > max_tiles:
        step = int(math.ceil(len(candidates) / max_tiles))
        candidates = candidates[::step]
    rng = random.Random(CONFIG["SEED"])
    rng.shuffle(candidates)
    return candidates


def mosaic_sample(tiles: dict, tx: int, ty: int) -> np.ndarray | None:
    """Stitch 4 tiles (2x2 of 256 px) into one TILE_SIZE sample, or None.

    Slippy tile grid: x = column (east), y = row (SOUTH), so the 512 block
    anchored at (tx, ty) is  [[(tx,ty) NW, (tx+1,ty) NE],
                              [(tx,ty+1) SW, (tx+1,ty+1) SE]]."""
    top = tiles.get((tx, ty)); top_r = tiles.get((tx + 1, ty))
    bot = tiles.get((tx, ty + 1)); bot_r = tiles.get((tx + 1, ty + 1))
    if any(t is None for t in (top, top_r, bot, bot_r)):
        return None
    row_t = np.concatenate([top, top_r], axis=1)
    row_b = np.concatenate([bot, bot_r], axis=1)
    return np.concatenate([row_t, row_b], axis=0)


def rasterize_roads(roads: list[dict], anchor: tuple[int, int],
                    tile_size: int, bucket: dict | None = None) -> Image.Image | None:
    """Paint one sample's road mask (white roads on black), roads in RED later."""
    tx, ty = anchor
    origin_px, origin_py = tx * TILE, ty * TILE
    img = Image.new("L", (tile_size, tile_size), 0)
    d = ImageDraw.Draw(img)
    if bucket is None:
        geoms = roads
    else:
        geoms = bucket.get(tx, []) + bucket.get(tx + 1, [])
    for item in geoms:
        pts = item["pts"]
        x0, y0 = pts[:, 0].min(), pts[:, 1].min()
        x1, y1 = pts[:, 0].max(), pts[:, 1].max()
        if x1 < origin_px or x0 > origin_px + tile_size or \
           y1 < origin_py or y0 > origin_py + tile_size:
            continue
        local = [(float(px - origin_px), float(py - origin_py)) for px, py in pts]
        d.line(local, fill=255, width=item["width_px"], joint="curve")
    arr = np.asarray(img)
    return img if arr.any() else None


def build_city_dataset(city: str, cfg: dict, max_tiles: int | None = None) -> dict:
    """Full pipeline for ONE city: polygon -> OSM roads -> imagery tiles -> masks.

    Returns dict(images=NxHxWx3 uint8, masks=NxHxW uint8 {0,255}, zoom, mpp).
    """
    t0 = time.time()
    print(f"\n=== [{city}] downloading data ===")
    polygon = get_city_polygon(city)
    lat_c, lon_c = polygon.centroid.y, polygon.centroid.x
    zoom = pick_zoom(lat_c, cfg["TARGET_M_PER_PIXEL"], cfg["FALLBACK_ZOOM"])
    mpp = m_per_pixel(lat_c, zoom)
    print(f"[scale] {city}: zoom={zoom}  ({mpp:.2f} m/px — fixed ground scale)")

    roads = fetch_osm_roads(polygon, mpp, zoom)
    anchors = enumerate_sample_anchors(polygon, zoom, max_tiles or cfg["MAX_TILES_PER_CITY"])
    print(f"[tiles] {len(anchors)} candidate tiles of {cfg['TILE_SIZE']}px")

    # ---- download the 2x2 256-tiles needed by every anchor (parallel, polite)
    need: set[tuple[int, int]] = set()
    for tx, ty in anchors:
        need.add((tx, ty)); need.add((tx + 1, ty))
        need.add((tx, ty + 1)); need.add((tx + 1, ty + 1))
    tiles: dict[tuple[int, int], np.ndarray] = {}
    failed = 0
    with ThreadPoolExecutor(max_workers=6) as ex:
        futs = {ex.submit(download_tile, zoom, x, y, cfg["IMAGERY_SOURCE"]): (x, y)
                for x, y in need}
        for fut in tqdm(list(futs), desc=f"[imagery] {city}", ncols=90):
            t = fut.result()
            if t is None:
                failed += 1
            else:
                tiles[futs[fut]] = t
    print(f"[imagery] {len(tiles)} ok, {failed} failed/skipped")

    # ---- rasterize masks (bucket roads by 256-tile column for speed)
    bucket: dict[int, list[dict]] = {}
    for item in roads:
        c0 = int(item["pts"][:, 0].min() // TILE)
        c1 = int(item["pts"][:, 0].max() // TILE)
        for c in range(c0, c1 + 1):
            bucket.setdefault(c, []).append(item)

    imgs, msks, anchors_used = [], [], []
    empty_kept = 0
    for tx, ty in anchors:
        photo = mosaic_sample(tiles, tx, ty)
        if photo is None:
            continue
        mask_img = rasterize_roads(roads, (tx, ty), cfg["TILE_SIZE"], bucket)
        mask = np.zeros((cfg["TILE_SIZE"], cfg["TILE_SIZE"]), dtype=np.uint8) \
            if mask_img is None else np.asarray(mask_img)
        frac = float((mask > 0).mean())
        if frac < cfg["MIN_ROAD_FRACTION"]:
            if empty_kept < cfg["MAX_EMPTY_TILES"]:
                empty_kept += 1
            else:
                continue
        imgs.append(photo)
        msks.append(mask)
        anchors_used.append((tx, ty))

    images = np.stack(imgs) if imgs else np.zeros((0,), np.uint8)
    masks = np.stack(msks) if msks else np.zeros((0,), np.uint8)
    print(f"[done] {city}: {len(images)} samples "
          f"(road>= {cfg['MIN_ROAD_FRACTION']:.2f} of pixels), "
          f"mean road fraction {float((masks > 0).mean()) * 100:.1f}%, "
          f"{time.time() - t0:.0f}s")
    return {"images": images, "masks": masks, "zoom": zoom, "mpp": mpp,
            "city": city, "anchors": anchors_used}


# =============================================================================
# 4. Dataset + augmentation
# =============================================================================
class RoadTileDataset(Dataset):
    def __init__(self, images: np.ndarray, masks: np.ndarray, augment: bool = True,
                 mean=None, std=None):
        assert len(images) == len(masks)
        self.images = images            # (N, H, W, 3) uint8
        self.masks = masks              # (N, H, W) uint8
        self.augment = augment
        # ImageNet stats when the pretrained ResNet encoder is active (wired in
        # main()); plain /255 scaling for the from-scratch U-Net.
        self.mean = np.asarray(mean if mean is not None else (0.0, 0.0, 0.0),
                               np.float32).reshape(1, 1, 3)
        self.std = np.asarray(std if std is not None else (1.0, 1.0, 1.0),
                              np.float32).reshape(1, 1, 3)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, i: int):
        img = self.images[i]
        msk = self.masks[i]
        if self.augment:
            k = random.randint(0, 3)
            img = np.rot90(img, k)
            msk = np.rot90(msk, k)
            if random.random() < 0.5:
                img = img[:, ::-1]
                msk = msk[:, ::-1]
            if random.random() < 0.5:
                img = img[::-1, :]
                msk = msk[::-1, :]
            c = random.uniform(0.85, 1.15)          # brightness
            b = random.uniform(-18, 18)             # contrast offset
            img = np.clip(img.astype(np.float32) * c + b, 0, 255).astype(np.uint8)
        x = (np.ascontiguousarray(img).astype(np.float32) / 255.0 - self.mean) / self.std
        x = torch.from_numpy(x).permute(2, 0, 1)
        y = torch.from_numpy(np.ascontiguousarray(msk)).float().unsqueeze(0) / 255.0
        return x, y


def split_train_val(pack: dict, val_frac: float, seed: int) -> tuple[dict, dict]:
    n = len(pack["images"])
    idx = list(range(n))
    random.Random(seed).shuffle(idx)
    n_val = min(max(1, int(round(n * val_frac))), max(0, n - 1))  # keep >=1 train tile
    vi, ti = idx[:n_val], idx[n_val:]
    def sub(ii):
        return {"images": pack["images"][ii], "masks": pack["masks"][ii],
                "zoom": pack["zoom"], "mpp": pack["mpp"], "city": pack["city"]}
    return sub(ti), sub(vi)


# =============================================================================
# 5. Model: compact U-Net
# =============================================================================
class DoubleConv(nn.Module):
    def __init__(self, cin: int, cout: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1, bias=False),
            nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1, bias=False),
            nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class Up(nn.Module):
    def __init__(self, ch_top: int, ch_skip: int, ch_out: int):
        super().__init__()
        self.reduce = nn.Conv2d(ch_top, ch_skip, 3, padding=1)
        self.conv = DoubleConv(ch_skip * 2, ch_out)

    def forward(self, x, skip):
        x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear",
                          align_corners=False)
        x = self.reduce(x)
        x = torch.cat([x, skip], dim=1)
        return self.conv(x)


class UNet(nn.Module):
    """4 downsamplings; ~2.6 M params at base=16 — comfortable on a free T4."""
    def __init__(self, in_ch: int = 3, base: int = 16):
        super().__init__()
        c1, c2, c3, c4, c5 = (base, base * 2, base * 4, base * 8, base * 16)
        self.e1 = DoubleConv(in_ch, c1)
        self.e2 = DoubleConv(c1, c2)
        self.e3 = DoubleConv(c2, c3)
        self.e4 = DoubleConv(c3, c4)
        self.pool = nn.MaxPool2d(2)
        self.bottom = DoubleConv(c4, c5)
        self.u1 = Up(c5, c4, c4)
        self.u2 = Up(c4, c3, c3)
        self.u3 = Up(c3, c2, c2)
        self.u4 = Up(c2, c1, c1)
        self.outc = nn.Conv2d(c1, 1, 1)

    def forward(self, x):
        s1 = self.e1(x)
        s2 = self.e2(self.pool(s1))
        s3 = self.e3(self.pool(s2))
        s4 = self.e4(self.pool(s3))
        b = self.bottom(self.pool(s4))
        x = self.u1(b, s4)
        x = self.u2(x, s3)
        x = self.u3(x, s2)
        x = self.u4(x, s1)
        return self.outc(x)


class ResNetUNet(nn.Module):
    """Same U-Net shape, but the encoder is torchvision's ResNet-18 with ImageNet
    weights. On ~1k tiles this is the difference between "usable in ~10 epochs"
    and "usable in ~40": the encoder already knows edges, textures and repetitive
    man-made structure — the decoder only learns to read roads out of them.

    Expects ImageNet-normalized input (cfg IMG_MEAN / IMG_STD, wired in main()).
    Logits are produced at half resolution and bilinearly upsampled x2 — at
    0.7 m/px roads are 6-9 px wide, so this costs almost nothing.
    """
    def __init__(self, pretrained: bool = True):
        super().__init__()
        import torchvision
        try:
            from torchvision.models import ResNet18_Weights
            weights = ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
        except ImportError:                      # torchvision < 0.13
            weights = "imagenet" if pretrained else None
        try:
            self.backbone = torchvision.models.resnet18(weights=weights)
        except Exception as e:                   # weight download failed
            raise RuntimeError(f"resnet18 ImageNet weights unavailable: {e}") from e
        self.u1 = Up(512, 256, 256)
        self.u2 = Up(256, 128, 128)
        self.u3 = Up(128, 64, 64)
        self.u4 = Up(64, 64, 64)
        self.outc = nn.Conv2d(64, 1, 1)

    def encoder_parameters(self):
        return self.backbone.parameters()

    def decoder_parameters(self):
        for m in (self.u1, self.u2, self.u3, self.u4, self.outc):
            yield from m.parameters()

    def forward(self, x):
        H, W = x.shape[-2:]
        r = self.backbone
        s1 = r.relu(r.bn1(r.conv1(x)))           # 64 ch, H/2
        s2 = r.layer1(r.maxpool(s1))             # 64 ch, H/4
        s3 = r.layer2(s2)                        # 128 ch, H/8
        s4 = r.layer3(s3)                        # 256 ch, H/16
        b = r.layer4(s4)                         # 512 ch, H/32
        d = self.u1(b, s4)
        d = self.u2(d, s3)
        d = self.u3(d, s2)
        d = self.u4(d, s1)
        z = self.outc(d)                         # logits at H/2
        return F.interpolate(z, size=(H, W), mode="bilinear", align_corners=False)


def build_model(cfg: dict) -> nn.Module:
    """Pretrained-encoder U-Net when possible, compact from-scratch U-Net otherwise."""
    if cfg.get("PRETRAINED_ENCODER", False):
        try:
            m = ResNetUNet(pretrained=True)
            n = sum(p.numel() for p in m.parameters())
            print(f"[model] ResNet-18 encoder (ImageNet weights) + U-Net decoder  "
                  f"({n / 1e6:.1f} M params)")
            return m
        except Exception as e:
            print(f"[warn] pretrained encoder unavailable -> compact from-scratch U-Net ({e})")
    m = UNet(base=cfg["UNET_BASE"])
    n = sum(p.numel() for p in m.parameters())
    print(f"[model] compact U-Net base={cfg['UNET_BASE']} from scratch  ({n / 1e6:.2f} M params)")
    return m


def segmentation_loss(logits: torch.Tensor, target: torch.Tensor,
                      pos_weight="auto", overlap_weight: float = 1.0,
                      mode: str = "dice", tversky_alpha: float = 0.7,
                      tversky_beta: float = 0.3, return_parts: bool = False):
    """BCE (class-balanced by default — roads are only ~5-10% of pixels) plus a
    region-overlap term: soft Dice (mode="dice") or Tversky (mode="tversky").

    Tversky = TP / (TP + a*FP + b*FN). With alpha > beta every FALSE POSITIVE
    (forest/house painted as road) costs more than a missed road — the lever
    against over-marking. alpha = beta = 0.5 reproduces plain Dice exactly.

    HOW TO READ THE NUMBER (verified numerically, see scripts/loss_floor_check.py):
      untrained net          total ~ 2.0-2.2   (p = 0.5 everywhere)
      decent model (Dice .6) total ~ 0.9-1.1
      great model  (Dice .8) total ~ 0.3-0.5
    The loss starts near 2.1 BY DESIGN (pos_weight up to 20 inflates the BCE
    term for the rare road pixels) and falls towards ~1.0 — it does not start
    near 0. Judge training by IoU / precision / recall, not the raw loss.
    """
    w = None
    if pos_weight == "auto":
        pos = target.sum()
        w = ((target.numel() - pos) / pos.clamp(min=1.0)).clamp(1.0, 20.0).reshape(())
    elif pos_weight is not None and pos_weight != 1.0:
        w = torch.tensor([float(pos_weight)], device=logits.device).reshape(())
    bce = F.binary_cross_entropy_with_logits(logits, target, pos_weight=w)
    p = torch.sigmoid(logits)
    tp = (p * target).sum()
    if mode == "tversky":
        fp = (p * (1.0 - target)).sum()
        fn = ((1.0 - p) * target).sum()
        overlap = 1.0 - (tp + 1.0) / (tp + tversky_alpha * fp + tversky_beta * fn + 1.0)
    else:                                   # plain soft Dice
        overlap = 1.0 - (2.0 * tp + 1.0) / (p.sum() + target.sum() + 1.0)
    loss = bce + overlap_weight * overlap
    if return_parts:      # for logging: one fused GPU->CPU transfer per step
        if w is None:
            w = torch.ones((), device=logits.device)
        parts = torch.stack([loss.detach(), bce.detach(),
                             (overlap_weight * overlap).detach(), w])
        return loss, parts
    return loss


@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader,
             device: torch.device) -> tuple[float, float, float, float]:
    """Thresholded IoU, Dice, precision and recall over a loader.

    precision = TP / predicted-positive   -> LOW precision = over-marking
                (forests/houses painted red) — the metric to watch for that.
    recall    = TP / true road pixels     -> LOW recall = missed roads.
    """
    model.eval()
    inter = pred_sum = t_sum = union = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
        prob = torch.sigmoid(model(xb))
        pred = (prob > 0.5).float()
        inter += float((pred * yb).sum())
        pred_sum += float(pred.sum())
        t_sum += float(yb.sum())
        union += float(((pred + yb) > 0).sum())
    iou = inter / union if union > 0 else 0.0
    dice = (2 * inter) / (pred_sum + t_sum) if (pred_sum + t_sum) > 0 else 0.0
    precision = inter / pred_sum if pred_sum > 0 else 0.0
    recall = inter / t_sum if t_sum > 0 else 0.0
    return iou, dice, precision, recall


def self_test_mosaic() -> None:
    """Offline check: mosaic_sample must assemble the 2x2 tile block in the
    correct geographic order (x = east column, y = SOUTH row)."""
    c = {"NW": (255, 0, 0), "NE": (0, 255, 0), "SW": (0, 0, 255), "SE": (255, 255, 0)}
    tiles = {(0, 0): np.full((256, 256, 3), c["NW"], np.uint8),
             (1, 0): np.full((256, 256, 3), c["NE"], np.uint8),
             (0, 1): np.full((256, 256, 3), c["SW"], np.uint8),
             (1, 1): np.full((256, 256, 3), c["SE"], np.uint8)}
    m = mosaic_sample(tiles, 0, 0)
    assert tuple(m[0, 0]) == c["NW"] and tuple(m[0, 511]) == c["NE"]
    assert tuple(m[511, 0]) == c["SW"] and tuple(m[511, 511]) == c["SE"]
    print("[self-test] mosaic assembly OK")


def self_test_model() -> None:
    torch.manual_seed(0)
    m = UNet(base=CONFIG["UNET_BASE"])
    x = torch.randn(1, 3, 64, 64)
    y = (torch.rand(1, 1, 64, 64) > 0.9).float()
    out = m(x)
    assert out.shape == y.shape
    loss = segmentation_loss(out, y)
    assert torch.isfinite(loss)
    loss.backward()
    od = out.detach()          # detached copy: no requires_grad warnings below
    loss2, parts = segmentation_loss(od, y, return_parts=True)
    assert parts.shape == (4,) and bool(torch.isfinite(parts).all())
    loss_tv = segmentation_loss(od, y, mode="tversky")
    assert torch.isfinite(loss_tv)
    loss_tv05 = segmentation_loss(od, y, mode="tversky",
                                  tversky_alpha=0.5, tversky_beta=0.5)
    # Tversky(.5,.5) == Dice exactly only without smoothing; with smooth=1 they
    # differ by ~(fp+fn)/D^2 (~1e-3 rel) — negligible at training batch sizes.
    assert abs(float(loss_tv05) - float(loss)) / (1.0 + float(loss)) < 1e-2
    n_params = sum(p.numel() for p in m.parameters())
    print(f"[self-test] model OK ({n_params / 1e6:.2f} M params, forward+backward on CPU)")
    try:
        import torchvision  # noqa: F401
    except Exception:
        print("[self-test] torchvision not available — ResNet encoder path untested here")
    else:
        m2 = ResNetUNet(pretrained=False)         # random init: no download in self-test
        z = m2(torch.randn(1, 3, 64, 64))
        assert tuple(z.shape) == (1, 1, 64, 64), tuple(z.shape)
        n2 = sum(p.numel() for p in m2.parameters())
        print(f"[self-test] ResNet-18 encoder U-Net OK ({n2 / 1e6:.1f} M params, random init)")


# =============================================================================
# 6. Training loop (time-budgeted, keeps the best checkpoint)
# =============================================================================
def train_model(model: nn.Module, train_loader: DataLoader, val_loader: DataLoader,
                cfg: dict, save_path: str) -> dict:
    model.to(DEVICE)
    if hasattr(model, "encoder_parameters"):      # pretrained encoder: gentle LR
        opt = torch.optim.AdamW(
            [{"params": list(model.encoder_parameters()),
              "lr": cfg.get("ENCODER_LR", 2.5e-4)},
             {"params": list(model.decoder_parameters()), "lr": cfg["LR"]}],
            weight_decay=cfg["WEIGHT_DECAY"])
        print(f"[train] LR groups: encoder {cfg.get('ENCODER_LR', 2.5e-4)} (pretrained "
              f"weights), decoder/new layers {cfg['LR']}")
    else:
        opt = torch.optim.AdamW(model.parameters(), lr=cfg["LR"],
                                weight_decay=cfg["WEIGHT_DECAY"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg["MAX_EPOCHS"])
    try:
        scaler = torch.amp.GradScaler("cuda", enabled=(cfg["AMP"] and DEVICE.type == "cuda"))
    except (TypeError, AttributeError):           # older torch
        scaler = torch.cuda.amp.GradScaler(enabled=(cfg["AMP"] and DEVICE.type == "cuda"))
    budget_s = cfg["TRAIN_TIME_BUDGET_MIN"] * 60
    t0 = time.time()
    mode = "tversky" if cfg.get("LOSS_MODE", "dice_bce") == "tversky_bce" else "dice"
    ov_name = "tversky" if mode == "tversky" else "dice"
    history = {"train_loss": [], "train_bce": [], "train_overlap": [],
               "pos_weight": [], "val_iou": [], "val_dice": [],
               "val_precision": [], "val_recall": []}
    best_iou = -1.0
    if mode == "tversky":
        print(f"[train] loss = BCE + Tversky(alpha={cfg['TVERSKY_ALPHA']}, "
              f"beta={cfg['TVERSKY_BETA']}): false positives (forest/house painted "
              "as road) cost more than missed roads. Starts near 2.0 — watch "
              "precision go UP while recall stays high.")
    else:
        print("[train] loss = BCE + Dice. Starts near 2.1, heads for ~1.0 — "
              "judge progress by IoU/precision/recall, not the loss value.")

    for epoch in range(1, cfg["MAX_EPOCHS"] + 1):
        model.train()
        sums, n_seen = np.zeros(4), 0          # [total, bce, overlap, pos_weight]
        for xb, yb in tqdm(train_loader, desc=f"epoch {epoch:02d}", ncols=90, leave=False):
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=(cfg["AMP"] and DEVICE.type == "cuda")):
                loss, parts = segmentation_loss(model(xb), yb, cfg["BCE_POS_WEIGHT"],
                                                cfg["OVERLAP_WEIGHT"], mode,
                                                cfg["TVERSKY_ALPHA"], cfg["TVERSKY_BETA"],
                                                return_parts=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            sums += np.asarray(parts.cpu().tolist())   # single sync for 4 values
            n_seen += 1
        sched.step()

        avg = sums / max(1, n_seen)
        iou, dice, prec, rec = evaluate(model, val_loader, DEVICE)
        history["train_loss"].append(avg[0])
        history["train_bce"].append(avg[1])
        history["train_overlap"].append(avg[2])
        history["pos_weight"].append(avg[3])
        history["val_iou"].append(iou)
        history["val_dice"].append(dice)
        history["val_precision"].append(prec)
        history["val_recall"].append(rec)
        marker = ""
        if iou > best_iou:
            best_iou = iou
            torch.save({"model": model.state_dict(), "epoch": epoch, "val_iou": iou,
                        "val_precision": prec, "val_recall": rec,
                        "config": cfg, "unet_base": cfg["UNET_BASE"],
                        "arch": type(model).__name__}, save_path)
            marker = "  <- saved"
        sim = max(0.0, 1 - avg[2])   # soft Dice score (dice mode) / Tversky index
        print(f"[epoch {epoch:02d}] loss={avg[0]:.3f} (bce {avg[1]:.3f} + {ov_name} {avg[2]:.3f})  "
              f"train sim~{sim:.3f}  "
              f"val IoU={iou:.3f} Dice={dice:.3f} P={prec:.3f} R={rec:.3f}  "
              f"elapsed={(time.time() - t0) / 60:.1f} min{marker}")

        if (time.time() - t0) > budget_s:
            print(f"[train] time budget {cfg['TRAIN_TIME_BUDGET_MIN']} min reached — stopping.")
            break

    # learning curves
    fig, ax = plt.subplots(1, 3, figsize=(13, 3.2))
    ax[0].plot(history["train_loss"]); ax[0].set_title(f"train loss (BCE+{ov_name})")
    ax[1].plot(history["train_bce"], label="BCE (pos_weight-balanced)")
    ax[1].plot(history["train_overlap"], label=f"{ov_name} loss")
    ax[1].set_title("loss components"); ax[1].legend(fontsize=8)
    ax[2].plot(history["val_iou"], label="IoU")
    ax[2].plot(history["val_dice"], label="Dice")
    ax[2].plot(history["val_precision"], label="precision")
    ax[2].plot(history["val_recall"], label="recall")
    ax[2].set_title("validation"); ax[2].legend(fontsize=8)
    for a in ax: a.set_xlabel("epoch"); a.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(os.path.join(cfg["SAVE_DIR"], "training_history.png"), dpi=130)
    plt.close(fig)
    print(f"[train] best val IoU = {best_iou:.3f}")
    return history


# =============================================================================
# 7. Inference on an UNSEEN city: stitch a big photo, paint roads red, IoU
# =============================================================================
def stitched_prediction(model: nn.Module, photo: np.ndarray,
                        cfg: dict, device: torch.device) -> np.ndarray:
    """Sliding 512-px windows (stride 256) with soft blending -> prob map.
    With cfg['PRED_TTA'] each window is predicted in 4 rotations and the
    results are rotated back and averaged — thin roads that a single
    orientation under-detects become visible (cheap recall boost)."""
    H, W = photo.shape[:2]
    s = torch.zeros((1, 1, H, W), dtype=torch.float32, device=device)
    c = torch.zeros((1, 1, H, W), dtype=torch.float32, device=device)
    ts = cfg["TILE_SIZE"]
    tta = 4 if cfg.get("PRED_TTA") else 1
    # same normalization the training tiles used (cfg IMG_MEAN/IMG_STD, which are
    # plain /255 defaults for the from-scratch U-Net)
    mean = np.asarray(cfg.get("IMG_MEAN", (0.0, 0.0, 0.0)), np.float32).reshape(1, 1, 3)
    std = np.asarray(cfg.get("IMG_STD", (1.0, 1.0, 1.0)), np.float32).reshape(1, 1, 3)
    photo_f = (photo.astype(np.float32) / 255.0 - mean) / std
    windows = [(y, x) for y in range(0, H - ts + 1, 256)
                       for x in range(0, W - ts + 1, 256)]
    model.eval()
    with torch.no_grad():
        for i in range(0, len(windows), cfg["PRED_BATCH"]):
            batch = windows[i:i + cfg["PRED_BATCH"]]
            xb = torch.stack([
                torch.from_numpy(np.ascontiguousarray(photo_f[y:y + ts, x:x + ts]))
                .permute(2, 0, 1)
                for y, x in batch]).to(device)
            acc = None
            for k in range(tta):
                # NB: torch.rot90 defaults to dims=(0,1) like numpy — that would
                # rotate the (batch, channels) plane and corrupt the batch shape.
                # Always pass the spatial plane explicitly.
                xk = torch.rot90(xb, k, dims=(-2, -1)) if k else xb
                pk = torch.sigmoid(model(xk))
                if k:
                    pk = torch.rot90(pk, -k, dims=(-2, -1))   # rotate prediction back
                acc = pk if acc is None else acc + pk
            probs = acc / tta
            for (y, x), p in zip(batch, probs):
                s[0, 0, y:y + ts, x:x + ts] += p[0]
                c[0, 0, y:y + ts, x:x + ts] += 1.0
    return (s / c.clamp(min=1)).squeeze().cpu().numpy()


def paint_roads_red(photo: np.ndarray, prob: np.ndarray, alpha: float,
                    thr: float = 0.5) -> tuple[np.ndarray, np.ndarray]:
    """Binary mask + photo with roads painted red."""
    mask = (prob > thr)
    red = np.zeros_like(photo, dtype=np.float32)
    red[..., 0] = 255
    out = photo.astype(np.float32) * (1 - alpha) + red * alpha * mask[..., None].astype(np.float32)
    overlay = np.clip(out, 0, 255).astype(np.uint8)
    red_mask = np.zeros_like(photo)
    red_mask[mask] = [255, 0, 0]
    return overlay, red_mask


def demo_unseen_city(model: nn.Module, city: str, cfg: dict, device: torch.device) -> float:
    """Stitch a large high-res photo of the city centre, paint roads red,
    save PNGs + 3-panel figure, and quantify IoU against OSM labels."""
    print(f"\n=== [unseen: {city}] inference & visualization ===")
    polygon = get_city_polygon(city)
    lat_c, lon_c = polygon.centroid.y, polygon.centroid.x
    zoom = pick_zoom(lat_c, cfg["TARGET_M_PER_PIXEL"], cfg["FALLBACK_ZOOM"])
    mpp = m_per_pixel(lat_c, zoom)

    ctx, cty = deg2num(lat_c, lon_c, zoom)
    ctx = int(round(ctx)) & ~1            # even 256-tile anchor
    cty = int(round(cty)) & ~1
    R = cfg["STITCH_RADIUS_TILES"]        # region = (2R+2) tiles wide
    tx0, ty0 = ctx - R, cty - R
    side = 2 * R + 2
    coords = [(tx0 + dx, ty0 + dy) for dy in range(side) for dx in range(side)]

    H = W = side * TILE
    photo = np.full((H, W, 3), 34, dtype=np.uint8)
    with ThreadPoolExecutor(max_workers=6) as ex:
        futs = {ex.submit(download_tile, zoom, x, y, cfg["IMAGERY_SOURCE"]): (x, y)
                for x, y in coords}
        for fut in tqdm(list(futs), desc=f"[imagery] {city}", ncols=90):
            x, y = futs[fut]
            t = fut.result()
            if t is not None:
                i, j = (x - tx0) * TILE, (y - ty0) * TILE
                photo[j:j + TILE, i:i + TILE] = t
    print(f"[region] {side}x{side} tiles -> {W}x{H} px @ {mpp:.2f} m/px")

    prob = stitched_prediction(model, photo, cfg, device)
    overlay, red_mask = paint_roads_red(photo, prob, cfg["OVERLAY_ALPHA"],
                                        thr=cfg["PRED_THRESHOLD"])

    safe = city.replace(" ", "_").replace(",", "")
    p_photo = os.path.join(cfg["SAVE_DIR"], f"{safe}_photo.png")
    p_mask = os.path.join(cfg["SAVE_DIR"], f"{safe}_roads_mask_red.png")
    p_over = os.path.join(cfg["SAVE_DIR"], f"{safe}_roads_red_overlay.png")
    Image.fromarray(photo).save(p_photo)
    Image.fromarray(red_mask).save(p_mask)
    Image.fromarray(overlay).save(p_over)

    fig, ax = plt.subplots(1, 3, figsize=(15, 5.2))
    ax[0].imshow(photo); ax[0].set_title(f"{city} — satellite photo")
    ax[1].imshow(red_mask); ax[1].set_title("predicted roads (red)")
    ax[2].imshow(overlay); ax[2].set_title("roads painted on the photo")
    for a in ax: a.axis("off")
    fig.tight_layout()
    fig.savefig(os.path.join(cfg["SAVE_DIR"], f"{safe}_result.png"), dpi=140)
    plt.close(fig)
    print(f"[saved] {p_photo}\n[saved] {p_mask}\n[saved] {p_over}")

    # ---- quantitative check against OSM ground truth on this unseen city
    pack = build_city_dataset(city, cfg, max_tiles=150)
    if len(pack["images"]) >= 8:
        ds = RoadTileDataset(pack["images"], pack["masks"], augment=False,
                             mean=cfg.get("IMG_MEAN"), std=cfg.get("IMG_STD"))
        loader = DataLoader(ds, batch_size=cfg["BATCH_SIZE"], num_workers=cfg["NUM_WORKERS"])
        iou, dice, prec, rec = evaluate(model, loader, device)
        print(f"[unseen {city}] IoU={iou:.3f}  Dice={dice:.3f}  "
              f"precision={prec:.3f}  recall={rec:.3f}  ({len(ds)} tiles)")
        return iou
    print(f"[unseen {city}] not enough labeled tiles for IoU — visual output only")
    return float("nan")


# =============================================================================
# 8. main()
# =============================================================================
def main() -> None:
    t_start = time.time()
    cfg = CONFIG
    os.makedirs(cfg["SAVE_DIR"], exist_ok=True)
    set_seed(cfg["SEED"])

    print("=" * 74)
    print(" ROAD SEGMENTATION U-NET  |  device:", DEVICE,
          f"({torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'CPU — will be slow!'})")
    print(f" train cities: {cfg['TRAIN_CITIES']}   unseen: {cfg['UNSEEN_CITIES']}")
    print(f" imagery: {cfg['IMAGERY_SOURCE']}   target scale: {cfg['TARGET_M_PER_PIXEL']} m/px")
    print(f" road labels: {cfg['ROADS_SOURCE']}   (Overpass API not needed)")
    print(f" loss: {cfg.get('LOSS_MODE', 'dice_bce')}   "
          f"demo paint threshold: {cfg['PRED_THRESHOLD']}")
    print(f" encoder: {'ResNet-18 (ImageNet-pretrained)' if cfg.get('PRETRAINED_ENCODER') else 'from scratch'}")
    print("=" * 74)

    assert cfg["TILE_SIZE"] % 256 == 0, "TILE_SIZE must be a multiple of 256"
    self_test_geo()
    self_test_mosaic()
    self_test_model()
    if cfg["IMAGERY_SOURCE"] == "osm":
        print("[warn] IMAGERY_SOURCE='osm' serves DRAWN map tiles (roads are pre-"
              "painted) — use 'esri' for honest satellite training.")

    # ---------- 0. model + input normalization -------------------------------
    # (built first: the dataset must know whether to use ImageNet statistics)
    model = build_model(cfg)
    if isinstance(model, ResNetUNet):
        cfg["IMG_MEAN"] = (0.485, 0.456, 0.406)   # ImageNet stats — must match
        cfg["IMG_STD"] = (0.229, 0.224, 0.225)    # the encoder's pretraining
        print("[model] input normalization: ImageNet mean/std")
    else:
        cfg["IMG_MEAN"] = (0.0, 0.0, 0.0)         # plain /255 for the from-scratch U-Net
        cfg["IMG_STD"] = (1.0, 1.0, 1.0)

    # ---------- 1. data -----------------------------------------------------
    train_packs, val_packs = [], []
    for city in cfg["TRAIN_CITIES"]:
        pack = build_city_dataset(city, cfg)
        tr, va = split_train_val(pack, cfg["VAL_SPLIT"], cfg["SEED"] + len(train_packs))
        train_packs.append(tr); val_packs.append(va)

    def to_loader(packs, shuffle):
        ds = RoadTileDataset(
            np.concatenate([p["images"] for p in packs]),
            np.concatenate([p["masks"] for p in packs]), augment=shuffle,
            mean=cfg.get("IMG_MEAN"), std=cfg.get("IMG_STD"))
        return DataLoader(ds, batch_size=cfg["BATCH_SIZE"], shuffle=shuffle,
                          num_workers=cfg["NUM_WORKERS"], pin_memory=(DEVICE.type == "cuda"),
                          drop_last=shuffle,
                          # keep workers alive between epochs: faster epochs and
                          # avoids the noisy "can only test a child process" GC spam
                          persistent_workers=(cfg["NUM_WORKERS"] > 0))

    train_loader = to_loader(train_packs, shuffle=True)
    val_loader = to_loader(val_packs, shuffle=False)
    print(f"\n[data] train tiles: {len(train_loader.dataset)}  "
          f"val tiles: {len(val_loader.dataset)}")

    # ---------- 2. training ----------------------------------------------------
    ckpt_path = os.path.join(cfg["SAVE_DIR"], "best_model.pt")
    train_model(model, train_loader, val_loader, cfg, ckpt_path)

    # ---------- 3. unseen cities ---------------------------------------------
    state = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(state["model"])
    print(f"\n[eval] loaded best checkpoint (epoch {state['epoch']}, "
          f"val IoU {state['val_iou']:.3f})")
    unseen_ious = []
    for city in cfg["UNSEEN_CITIES"]:
        unseen_ious.append((city, demo_unseen_city(model, city, cfg, DEVICE)))

    # ---------- 4. summary ----------------------------------------------------
    print("\n" + "=" * 74)
    print(" DONE in", f"{(time.time() - t_start) / 60:.1f} min")
    print(" Outputs in", cfg["SAVE_DIR"])
    print("   best_model.pt                     trained U-Net checkpoint")
    print("   training_history.png              loss / IoU curves")
    for city, iou in unseen_ious:
        safe = city.replace(" ", "_").replace(",", "")
        extra = f" ({iou:.3f} IoU vs OSM)" if iou == iou else ""
        print(f"   {safe}_result.png                 photo | red roads | overlay")
        print(f"   {safe}_roads_red_overlay.png      full-res red overlay{extra}")
    print(" Imagery (c) Esri World Imagery | labels (c) OpenStreetMap contributors")
    print("=" * 74)


if __name__ == "__main__":
    main()

In [ ]:

import math
import os
import subprocess
import sys
import time
import traceback
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import requests

requests.packages.urllib3.disable_warnings()

try:
    import torch
except ImportError as e:
    raise ImportError(
        "PyTorch not found — on Colab it is preinstalled. Locally: pip install torch"
    ) from e

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True

# =============================================================================
# CONFIG — edit freely, no argparse
# =============================================================================
CONFIG = {
    "CKPT_PATH": None,        # None = auto-find best_model.pt (see find_checkpoint)
    "DEFAULT_CENTER": (58.0105, 56.2502),   # Perm — where the last demo looked
    "MAP_ZOOM": 13,           # display zoom of the picker map (satellite basemap)
    "REGION_TILES": 8,        # default region: 8x8 tiles of 256 px = 2048 px side
    "MAX_REGION_TILES": 16,   # cap: 16 -> 4096 px side (~2-3 min on a T4 with TTA)
    "SAVE_DIR": "/content/road_seg_outputs",  # PNGs go here (same folder as training)
}

# =============================================================================
# imagery tiles (verbatim from road_segmentation_colab.py)
# =============================================================================
_UA = {"User-Agent": "road-segmentation-colab/1.0 (student project)"}

_session = requests.Session()
_session.headers.update(_UA)
_session.mount("https://", requests.adapters.HTTPAdapter(max_retries=2))

IMAGERY_URLS = {
    "esri": "https://server.arcgisonline.com/ArcGIS/rest/services/"
            "World_Imagery/MapServer/tile/{z}/{y}/{x}",
    "osm":  "https://tile.openstreetmap.org/{z}/{x}/{y}.png",
}


def download_tile(z: int, x: int, y: int, source: str = "esri") -> np.ndarray | None:
    """Download one 256x256 imagery tile -> HxWx3 uint8 (or None on failure)."""
    url = IMAGERY_URLS[source].format(z=z, x=x, y=y)
    try:
        r = _session.get(url, timeout=40)
        if r.status_code != 200 or len(r.content) < 500:
            return None
        from io import BytesIO
        img = Image.open(BytesIO(r.content)).convert("RGB")
        if img.size != (TILE, TILE):
            img = img.resize((TILE, TILE))
        return np.asarray(img, dtype=np.uint8)
    except Exception:
        return None


# =============================================================================
# geo / Web-Mercator tile math (verbatim from road_segmentation_colab.py)
# =============================================================================
EARTH_R = 6378137.0
TILE = 256                          # standard tile size in px


def deg2num(lat_deg: float, lon_deg: float, zoom: int) -> tuple[float, float]:
    """lat/lon -> fractional (xtile, ytile) on the 256-px slippy grid."""
    n = 2 ** zoom
    lat = math.radians(max(min(lat_deg, 85.05112878), -85.05112878))
    x = (lon_deg + 180.0) / 360.0 * n
    y = (1.0 - math.log(math.tan(lat) + 1.0 / math.cos(lat)) / math.pi) / 2.0 * n
    return x, y


def num2deg(x: float, y: float, zoom: int) -> tuple[float, float]:
    """fractional (xtile, ytile) -> lat/lon of the tile's NW corner."""
    n = 2 ** zoom
    lon = x / n * 360.0 - 180.0
    lat = math.degrees(math.atan(math.sinh(math.pi * (1.0 - 2.0 * y / n))))
    return lat, lon


def m_per_pixel(lat_deg: float, zoom: int) -> float:
    """Ground resolution (m/px) of a 256-px tile at this latitude and zoom."""
    return 156543.03392 * math.cos(math.radians(lat_deg)) / (2 ** zoom)


def pick_zoom(lat_deg: float, target_mpp: float | None, fallback: int = 17,
              zmin: int = 16, zmax: int = 19) -> int:
    """Choose the integer zoom whose ground resolution is closest to target.

    This is what keeps the FIXED IMAGE SCALE across cities at different
    latitudes (m/px depends on latitude for a constant zoom).
    """
    if target_mpp is None:
        return fallback
    return min(range(zmin, zmax + 1),
               key=lambda z: abs(m_per_pixel(lat_deg, z) - target_mpp))


# =============================================================================
# model: compact U-Net (+ optional ImageNet-pretrained ResNet-18 encoder)
# verbatim from road_segmentation_colab.py — state dicts are interchangeable
# =============================================================================
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image


class DoubleConv(nn.Module):
    def __init__(self, cin: int, cout: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1, bias=False),
            nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1, bias=False),
            nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class Up(nn.Module):
    def __init__(self, ch_top: int, ch_skip: int, ch_out: int):
        super().__init__()
        self.reduce = nn.Conv2d(ch_top, ch_skip, 3, padding=1)
        self.conv = DoubleConv(ch_skip * 2, ch_out)

    def forward(self, x, skip):
        x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear",
                          align_corners=False)
        x = self.reduce(x)
        x = torch.cat([x, skip], dim=1)
        return self.conv(x)


class UNet(nn.Module):
    """4 downsamplings; ~2.6 M params at base=16 — comfortable on a free T4."""
    def __init__(self, in_ch: int = 3, base: int = 16):
        super().__init__()
        c1, c2, c3, c4, c5 = (base, base * 2, base * 4, base * 8, base * 16)
        self.e1 = DoubleConv(in_ch, c1)
        self.e2 = DoubleConv(c1, c2)
        self.e3 = DoubleConv(c2, c3)
        self.e4 = DoubleConv(c3, c4)
        self.pool = nn.MaxPool2d(2)
        self.bottom = DoubleConv(c4, c5)
        self.u1 = Up(c5, c4, c4)
        self.u2 = Up(c4, c3, c3)
        self.u3 = Up(c3, c2, c2)
        self.u4 = Up(c2, c1, c1)
        self.outc = nn.Conv2d(c1, 1, 1)

    def forward(self, x):
        s1 = self.e1(x)
        s2 = self.e2(self.pool(s1))
        s3 = self.e3(self.pool(s2))
        s4 = self.e4(self.pool(s3))
        b = self.bottom(self.pool(s4))
        x = self.u1(b, s4)
        x = self.u2(x, s3)
        x = self.u3(x, s2)
        x = self.u4(x, s1)
        return self.outc(x)


class ResNetUNet(nn.Module):
    """Same U-Net shape, but the encoder is torchvision's ResNet-18 with ImageNet
    weights. On ~1k tiles this is the difference between "usable in ~10 epochs"
    and "usable in ~40": the encoder already knows edges, textures and repetitive
    man-made structure — the decoder only learns to read roads out of them.

    Expects ImageNet-normalized input (cfg IMG_MEAN / IMG_STD, wired in main()).
    Logits are produced at half resolution and bilinearly upsampled x2 — at
    0.7 m/px roads are 6-9 px wide, so this costs almost nothing.
    """
    def __init__(self, pretrained: bool = True):
        super().__init__()
        import torchvision
        try:
            from torchvision.models import ResNet18_Weights
            weights = ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
        except ImportError:                      # torchvision < 0.13
            weights = "imagenet" if pretrained else None
        try:
            self.backbone = torchvision.models.resnet18(weights=weights)
        except Exception as e:                   # weight download failed
            raise RuntimeError(f"resnet18 ImageNet weights unavailable: {e}") from e
        self.u1 = Up(512, 256, 256)
        self.u2 = Up(256, 128, 128)
        self.u3 = Up(128, 64, 64)
        self.u4 = Up(64, 64, 64)
        self.outc = nn.Conv2d(64, 1, 1)

    def encoder_parameters(self):
        return self.backbone.parameters()

    def decoder_parameters(self):
        for m in (self.u1, self.u2, self.u3, self.u4, self.outc):
            yield from m.parameters()

    def forward(self, x):
        H, W = x.shape[-2:]
        r = self.backbone
        s1 = r.relu(r.bn1(r.conv1(x)))           # 64 ch, H/2
        s2 = r.layer1(r.maxpool(s1))             # 64 ch, H/4
        s3 = r.layer2(s2)                        # 128 ch, H/8
        s4 = r.layer3(s3)                        # 256 ch, H/16
        b = r.layer4(s4)                         # 512 ch, H/32
        d = self.u1(b, s4)
        d = self.u2(d, s3)
        d = self.u3(d, s2)
        d = self.u4(d, s1)
        z = self.outc(d)                         # logits at H/2
        return F.interpolate(z, size=(H, W), mode="bilinear", align_corners=False)


# =============================================================================
# inference: sliding-window TTA prediction + red painting (verbatim copies)
# =============================================================================
def stitched_prediction(model: nn.Module, photo: np.ndarray,
                        cfg: dict, device: torch.device) -> np.ndarray:
    """Sliding 512-px windows (stride 256) with soft blending -> prob map.
    With cfg['PRED_TTA'] each window is predicted in 4 rotations and the
    results are rotated back and averaged — thin roads that a single
    orientation under-detects become visible (cheap recall boost)."""
    H, W = photo.shape[:2]
    s = torch.zeros((1, 1, H, W), dtype=torch.float32, device=device)
    c = torch.zeros((1, 1, H, W), dtype=torch.float32, device=device)
    ts = cfg["TILE_SIZE"]
    tta = 4 if cfg.get("PRED_TTA") else 1
    # same normalization the training tiles used (cfg IMG_MEAN/IMG_STD, which are
    # plain /255 defaults for the from-scratch U-Net)
    mean = np.asarray(cfg.get("IMG_MEAN", (0.0, 0.0, 0.0)), np.float32).reshape(1, 1, 3)
    std = np.asarray(cfg.get("IMG_STD", (1.0, 1.0, 1.0)), np.float32).reshape(1, 1, 3)
    photo_f = (photo.astype(np.float32) / 255.0 - mean) / std
    windows = [(y, x) for y in range(0, H - ts + 1, 256)
                       for x in range(0, W - ts + 1, 256)]
    model.eval()
    with torch.no_grad():
        for i in range(0, len(windows), cfg["PRED_BATCH"]):
            batch = windows[i:i + cfg["PRED_BATCH"]]
            xb = torch.stack([
                torch.from_numpy(np.ascontiguousarray(photo_f[y:y + ts, x:x + ts]))
                .permute(2, 0, 1)
                for y, x in batch]).to(device)
            acc = None
            for k in range(tta):
                # NB: torch.rot90 defaults to dims=(0,1) like numpy — that would
                # rotate the (batch, channels) plane and corrupt the batch shape.
                # Always pass the spatial plane explicitly.
                xk = torch.rot90(xb, k, dims=(-2, -1)) if k else xb
                pk = torch.sigmoid(model(xk))
                if k:
                    pk = torch.rot90(pk, -k, dims=(-2, -1))   # rotate prediction back
                acc = pk if acc is None else acc + pk
            probs = acc / tta
            for (y, x), p in zip(batch, probs):
                s[0, 0, y:y + ts, x:x + ts] += p[0]
                c[0, 0, y:y + ts, x:x + ts] += 1.0
    return (s / c.clamp(min=1)).squeeze().cpu().numpy()


def paint_roads_red(photo: np.ndarray, prob: np.ndarray, alpha: float,
                    thr: float = 0.5) -> tuple[np.ndarray, np.ndarray]:
    """Binary mask + photo with roads painted red."""
    mask = (prob > thr)
    red = np.zeros_like(photo, dtype=np.float32)
    red[..., 0] = 255
    out = photo.astype(np.float32) * (1 - alpha) + red * alpha * mask[..., None].astype(np.float32)
    overlay = np.clip(out, 0, 255).astype(np.uint8)
    red_mask = np.zeros_like(photo)
    red_mask[mask] = [255, 0, 0]
    return overlay, red_mask


# =============================================================================
# checkpoint handling
# =============================================================================
def find_checkpoint(cfg: dict) -> str | None:
    """Return the first existing checkpoint path, or None (with a hint)."""
    candidates = []
    if cfg["CKPT_PATH"]:
        candidates.append(cfg["CKPT_PATH"])
    candidates += [
        os.path.join(cfg["SAVE_DIR"], "best_model.pt"),
        "/content/road_seg_outputs/best_model.pt",
        "/content/drive/MyDrive/road_seg_outputs/best_model.pt",  # if Drive mounted
        os.path.join(_HERE, "road_seg_outputs", "best_model.pt"),
        os.path.join(_HERE, "best_model.pt"),
    ]
    for c in candidates:
        if c and os.path.isfile(c):
            print(f"[ckpt] using {c}")
            return c
    print("[ckpt] best_model.pt not found. Looked at:")
    for c in candidates:
        print(f"         - {c}")
    print("[ckpt] the map tool only READS the checkpoint — it never retrains.")
    print("[ckpt] point CONFIG['CKPT_PATH'] at your existing best_model.pt")
    print("[ckpt] (e.g. mounted from Drive), or upload the file to /content/.")
    return None


def load_model_and_cfg(ckpt_path: str):
    """Rebuild the exact architecture from the checkpoint and load its weights.

    Returns (model, cfg) — cfg is the TRAINING config read from the checkpoint,
    so ground scale, thresholds, TTA and input normalization automatically
    match what the model saw during training.
    """
    state = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    cfg = dict(state["config"])                     # full training CONFIG
    cfg["SAVE_DIR"] = CONFIG["SAVE_DIR"]            # ...but save next to the ckpt
    arch = state.get("arch", "UNet")
    if arch == "ResNetUNet":
        model = ResNetUNet(pretrained=False)        # same architecture, no download
    else:
        model = UNet(base=state.get("unet_base", cfg.get("UNET_BASE", 16)))
    model.load_state_dict(state["model"])
    model.to(DEVICE).eval()
    extra = state.get("val_iou")
    print(f"[ckpt] loaded {arch} (epoch {state.get('epoch')}"
          + (f", val IoU {extra:.3f}" if extra else "")
          + f") on {DEVICE}")
    print(f"[cfg]  scale {cfg['TARGET_M_PER_PIXEL']} m/px | paint threshold "
          f"{cfg['PRED_THRESHOLD']} | TTA {'on' if cfg.get('PRED_TTA') else 'off'}")
    return model, cfg

# =============================================================================
# region fetch + inference
# =============================================================================
def fetch_region(lat: float, lon: float, side_tiles: int, cfg: dict):
    """Stitch a (side_tiles x side_tiles) 256-px region around (lat, lon) at the
    TRAINING ground scale. Returns (photo, zoom, mpp)."""
    zoom = pick_zoom(lat, cfg["TARGET_M_PER_PIXEL"], cfg["FALLBACK_ZOOM"])
    mpp = m_per_pixel(lat, zoom)
    ctx, cty = deg2num(lat, lon, zoom)
    ctx, cty = int(round(ctx)) & ~1, int(round(cty)) & ~1   # even anchor (as in training)
    tx0, ty0 = ctx - side_tiles // 2, cty - side_tiles // 2
    coords = [(tx0 + dx, ty0 + dy) for dy in range(side_tiles) for dx in range(side_tiles)]

    tiles: dict[tuple[int, int], np.ndarray] = {}
    ok = 0
    with ThreadPoolExecutor(max_workers=6) as ex:
        futs = {ex.submit(download_tile, zoom, x, y, cfg["IMAGERY_SOURCE"]): (x, y)
                for x, y in coords}
        for i, fut in enumerate(list(futs), 1):
            t = fut.result()
            if t is not None:
                tiles[futs[fut]] = t
                ok += 1
            if i % max(1, len(coords) // 8) == 0 or i == len(coords):
                print(f"[imagery] {i}/{len(coords)} tiles ({ok} ok)", flush=True)
    if not tiles:
        raise RuntimeError("no imagery tiles downloaded — try another place")

    H = W = side_tiles * TILE
    photo = np.full((H, W, 3), 34, dtype=np.uint8)
    for (x, y), t in tiles.items():
        i, j = (x - tx0) * TILE, (y - ty0) * TILE
        photo[j:j + TILE, i:i + TILE] = t
    return photo, zoom, mpp


def process_place(model, cfg: dict, lat: float, lon: float, side_tiles: int) -> None:
    """Fetch the region around (lat, lon), predict, paint red, save + show."""
    side_tiles = max(2, min(side_tiles, CONFIG["MAX_REGION_TILES"]))
    t0 = time.time()
    print(f"\n[process] {lat:.5f}, {lon:.5f}  region {side_tiles}x{side_tiles} 256-px tiles")
    photo, zoom, mpp = fetch_region(lat, lon, side_tiles, cfg)
    side_px = side_tiles * TILE
    print(f"[scale]   zoom={zoom} ({mpp:.2f} m/px — same scale as training)  "
          f"{side_px}x{side_px} px  (~{side_px * mpp / 1000:.1f} km side)")

    n_windows = len([(y, x) for y in range(0, side_px - cfg["TILE_SIZE"] + 1, 256)
                             for x in range(0, side_px - cfg["TILE_SIZE"] + 1, 256)])
    tta = 4 if cfg.get("PRED_TTA") else 1
    print(f"[predict] {n_windows} windows x {tta} TTA rotations ...", flush=True)
    prob = stitched_prediction(model, photo, cfg, DEVICE)

    overlay, red_mask = paint_roads_red(photo, prob, cfg["OVERLAY_ALPHA"],
                                        thr=cfg["PRED_THRESHOLD"])
    cov = float((prob > cfg["PRED_THRESHOLD"]).mean()) * 100
    print(f"[done]    road coverage {cov:.1f}% of the image  ({time.time() - t0:.0f}s)")

    safe = f"map_{lat:.4f}_{lon:.4f}_{side_px}px"
    import matplotlib.pyplot as plt
    paths = {k: os.path.join(cfg["SAVE_DIR"], f"{safe}_{k}.png")
             for k in ("photo", "roads_mask_red", "roads_red_overlay")}
    Image.fromarray(photo).save(paths["photo"])
    Image.fromarray(red_mask).save(paths["roads_mask_red"])
    Image.fromarray(overlay).save(paths["roads_red_overlay"])

    fig, ax = plt.subplots(1, 3, figsize=(15, 5.2))
    ax[0].imshow(photo); ax[0].set_title("satellite photo")
    ax[1].imshow(red_mask); ax[1].set_title("predicted roads (red)")
    ax[2].imshow(overlay); ax[2].set_title("roads painted on the photo")
    for a in ax:
        a.axis("off")
    fig.tight_layout()
    fig.savefig(os.path.join(cfg["SAVE_DIR"], f"{safe}_result.png"), dpi=140)
    plt.show()
    plt.close(fig)
    print(f"[saved]   {paths['photo']}\n[saved]   {paths['roads_mask_red']}"
          f"\n[saved]   {paths['roads_red_overlay']}\n"
          f"[saved]   {os.path.join(cfg['SAVE_DIR'], f'{safe}_result.png')}")

# =============================================================================
# UI: interactive map (ipyleaflet) -> plain widgets -> headless fallback
# =============================================================================
def _enable_colab_widgets() -> None:
    try:
        from google.colab import output
        output.enable_custom_widget_manager()
    except Exception:
        pass  # not on Colab — widgets still work in a local Jupyter


def _side_km_text(cfg: dict, lat: float, tiles: int) -> str:
    zoom = pick_zoom(lat, cfg["TARGET_M_PER_PIXEL"], cfg["FALLBACK_ZOOM"])
    mpp = m_per_pixel(lat, zoom)
    px = tiles * TILE
    return f"{px * mpp / 1000:.1f} km"


def run_ui(model, cfg: dict) -> None:
    _enable_colab_widgets()
    W, display = None, None
    try:
        import importlib.util
        if importlib.util.find_spec("ipyleaflet") is None:
            print("[setup] installing missing package: ipyleaflet ...")
            subprocess.check_call([sys.executable, "-m", "pip", "install",
                                   "--quiet", "ipyleaflet"])
        import ipywidgets as W
        from IPython.display import display
    except Exception as e:
        print("[ui] widget backend unavailable:", e)

    # ---------- full interactive satellite map (ipyleaflet) ------------------
    # NB: ipyleaflet.basemaps is deliberately NOT used. `from ipyleaflet import
    # ..., basemaps` can bind the earlier names and THEN raise on `basemaps`
    # (partial import binding), which crashed run_ui with UnboundLocalError.
    # A plain TileLayer built from the Esri World Imagery URL works on every
    # ipyleaflet version Colab has ever shipped — and any unexpected error
    # below now falls back to the plain lat/lon boxes instead of crashing.
    if W is not None:
        try:
            _run_leaflet_ui(model, cfg, W, display)
            return
        except Exception as e:
            print("[ui] interactive satellite map unavailable:", repr(e))
            print("[ui] falling back to plain lat/lon boxes instead.")

    # ---------- no leaflet: plain coordinate widgets ------------------------
    if W is not None:
        print("[ui] using lat/lon boxes (ipyleaflet not importable here).")
        lat_box = W.FloatText(value=CONFIG["DEFAULT_CENTER"][0], description="lat")
        lon_box = W.FloatText(value=CONFIG["DEFAULT_CENTER"][1], description="lon")
        slider = W.IntSlider(value=min(CONFIG["REGION_TILES"], CONFIG["MAX_REGION_TILES"]),
                             min=4, max=CONFIG["MAX_REGION_TILES"],
                             step=1, description="region tiles")
        btn = W.Button(description="Process this place", button_style="danger")
        out = W.Output(layout=W.Layout(border="1px solid #999", max_height="640px"))

        def on_btn(_):
            btn.disabled = True
            with out:
                out.clear_output()
                try:
                    process_place(model, cfg, lat_box.value, lon_box.value, slider.value)
                except Exception:
                    traceback.print_exc()
            btn.disabled = False

        btn.on_click(on_btn)
        display(W.VBox([lat_box, lon_box, slider, btn, out]))
        return

    # ---------- headless fallback (e.g. `!python` run) ----------------------
    lat, lon = CONFIG["DEFAULT_CENTER"]
    print("[ui] no widget backend available — processing CONFIG['DEFAULT_CENTER'] "
          f"({lat}, {lon}) headlessly and saving PNGs.")
    process_place(model, cfg, lat, lon, CONFIG["REGION_TILES"])


def _run_leaflet_ui(model, cfg: dict, W, display) -> None:
    """Build the click-a-place map UI. Raises only if ipyleaflet itself is
    unusable; run_ui() catches that and falls back to coordinate widgets."""
    import ipyleaflet as LFL

    center = tuple(CONFIG["DEFAULT_CENTER"])
    esri = LFL.TileLayer(
        url=IMAGERY_URLS["esri"], max_zoom=19,
        attribution="Esri, Maxar, Earthstar Geographics",
        name="Esri satellite")
    m = LFL.Map(center=center, zoom=CONFIG["MAP_ZOOM"], basemap=esri,
                scroll_wheel_zoom=True)
    try:                                        # toggleable OSM reference layer
        m.add_layer(LFL.TileLayer(url=IMAGERY_URLS["osm"], max_zoom=19,
                                  name="OSM streets (reference)", opacity=0.45))
        m.add_control(LFL.LayersControl(position="topright"))
    except Exception:
        pass                                    # older ipyleaflet without LayersControl
    marker = LFL.Marker(location=center, draggable=True)
    m.add_layer(marker)

    info = W.Label()
    slider = W.IntSlider(value=min(CONFIG["REGION_TILES"], CONFIG["MAX_REGION_TILES"]),
                         min=4, max=CONFIG["MAX_REGION_TILES"],
                         step=1, description="region tiles")
    btn = W.Button(description="Process this place", button_style="danger",
                   tooltip="fetch imagery at training scale and paint roads red")
    out = W.Output(layout=W.Layout(border="1px solid #999", max_height="640px",
                                   overflow="auto"))

    def refresh_info(_=None):
        lat, lon = marker.location
        info.value = (f"marker: {lat:.5f}, {lon:.5f}   |   region "
                      f"{slider.value}x{slider.value} tiles = "
                      f"{slider.value * 256} px (~"
                      f"{_side_km_text(cfg, lat, slider.value)} side)   "
                      "<- click the map to move the marker")

    def on_map_click(**kw):
        coords = kw.get("coordinates")
        if kw.get("type") == "click" and coords:
            marker.location = tuple(coords)
            refresh_info()

    m.on_interaction(on_map_click)
    marker.observe(refresh_info, names="location")
    slider.observe(refresh_info, names="value")

    def on_btn(_):
        btn.disabled = True
        with out:
            out.clear_output()
            try:
                lat, lon = marker.location
                process_place(model, cfg, lat, lon, slider.value)
            except Exception:
                traceback.print_exc()
        btn.disabled = False

    btn.on_click(on_btn)
    refresh_info()
    display(W.VBox([info, slider, btn, m, out]))


# =============================================================================
_HERE = (os.path.dirname(os.path.abspath(__file__))
         if "__file__" in globals() else "/content")


def main() -> None:
    print("=" * 74)
    print(" ROAD SEGMENTATION — INTERACTIVE MAP DEMO  (self-contained)")
    print(" device:", DEVICE, "  checkpoint folder:", CONFIG["SAVE_DIR"])
    print("=" * 74)
    os.makedirs(CONFIG["SAVE_DIR"], exist_ok=True)
    ckpt = find_checkpoint(CONFIG)
    if ckpt is None:
        return
    model, cfg = load_model_and_cfg(ckpt)
    run_ui(model, cfg)


if __name__ == "__main__":
    main()